# Step 2: a label-free correction, plus the missing control

Step 1 found: INT8 reuse is safe; pruning WITHOUT fine-tuning inflates APS sets (reuse overcovers); pruning WITH fine-tuning flips the sign (reuse undercovers, sets shrink). INT4 was seed-dependent and tied to accuracy collapse.

**Two additions**
1. **Control `fp32+ft`:** fine-tune the *unpruned* model the same 2 epochs. If it also undercovers under reuse, the effect is caused by extra training, not by compression. Without this control the pruned+ft result cannot be attributed to compression.
2. **`sizematch` (label-free):** using only the unlabeled calibration inputs, choose the threshold of the compressed model so its mean prediction-set size equals the mean set size the full-precision model produced with its calibrated threshold. No labels are used. **This is a hypothesis with no finite-sample guarantee**; it is evaluated empirically against `reuse` (stale threshold) and `recal` (needs labels).

Run all cells; check `smoke: False`. Results are saved per dataset to OUT_DIR. Use 5 seeds.

In [ ]:
!pip install -q medmnist scipy
import os, sys, copy, math, time, glob, itertools
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F

# ---- CONFIG --
SMOKE = False                      
DATASETS = ["dermamnist", "bloodmnist", "pathmnist"]   
SEEDS = [0, 1, 2, 3, 4]
EPOCHS = {"pathmnist": 8, "dermamnist": 15, "bloodmnist": 12}
FT_EPOCHS = 2                      # fine-tune epochs after pruning
N_SPLITS = 20                      
ALPHA = 0.1
USE_DRIVE = False                 
OUT_DIR = "/content/step2_out"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if USE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive"); OUT_DIR = "/content/drive/MyDrive/step2_out"
os.makedirs(OUT_DIR, exist_ok=True)
print("device:", DEVICE, "| smoke:", SMOKE, "| datasets:", DATASETS, "| seeds:", SEEDS, "| out:", OUT_DIR)
if DEVICE == "cpu": print("WARNING: CPU only. Switch to GPU (Runtime > Change runtime type) or run one dataset and one seed.")

def set_seed(s):
    np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

## Code

In [ ]:
# %%
def lac_all(probs):
    """Score of every label: 1 - p(label). probs: (n, K)."""
    return 1.0 - probs

def aps_all(probs):
    """Deterministic APS score of every label: total prob mass of labels ranked at or above it."""
    order = np.argsort(-probs, axis=1)
    cum = np.cumsum(np.take_along_axis(probs, order, axis=1), axis=1)
    ranks = np.argsort(order, axis=1)
    return np.take_along_axis(cum, ranks, axis=1)

SCORES = {"LAC": lac_all, "APS": aps_all}

def conformal_q(true_scores, alpha=ALPHA):
    """Finite-sample split-conformal quantile. Returns inf if the calibration set is too small."""
    n = len(true_scores)
    k = int(math.ceil((n + 1) * (1 - alpha)))
    if n == 0 or k > n:
        return np.inf
    return float(np.sort(true_scores)[k - 1])

def summarize(sets, y, min_class_n=10):
    covered = sets[np.arange(len(y)), y]
    cls = [covered[y == c].mean() for c in np.unique(y) if (y == c).sum() >= min_class_n]
    return dict(coverage=float(covered.mean()), size=float(sets.sum(1).mean()),
                worst_class_cov=float(min(cls)) if cls else np.nan)


In [ ]:
# %%
class Block(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.b1 = nn.BatchNorm2d(cout)
        self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False);     self.b2 = nn.BatchNorm2d(cout)
        self.sc = None
        if stride != 1 or cin != cout:
            self.sc = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
    def forward(self, x):
        o = F.relu(self.b1(self.c1(x))); o = self.b2(self.c2(o))
        return F.relu(o + (x if self.sc is None else self.sc(x)))

class ResNet18Small(nn.Module):
    """ResNet-18 layout (2,2,2,2 blocks) with a 3x3 stem, suited to 28x28 images."""
    def __init__(self, num_classes, width=64):
        super().__init__()
        w = width
        self.stem = nn.Sequential(nn.Conv2d(3, w, 3, 1, 1, bias=False), nn.BatchNorm2d(w), nn.ReLU())
        cfg, layers, cin = [(w, 1), (w, 1), (2*w, 2), (2*w, 1), (4*w, 2), (4*w, 1), (8*w, 2), (8*w, 1)], [], w
        for cout, s in cfg:
            layers.append(Block(cin, cout, s)); cin = cout
        self.layers = nn.Sequential(*layers); self.fc = nn.Linear(cin, num_classes)
    def forward(self, x):
        x = self.layers(self.stem(x))
        return self.fc(F.adaptive_avg_pool2d(x, 1).flatten(1))

def train_model(model, X, y, epochs, lr=2e-3, bs=256, seed=0):
    set_seed(seed); model.to(DEVICE).train()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * math.ceil(len(y) / bs))
    Xt, yt = torch.as_tensor(X), torch.as_tensor(y).long()
    for ep in range(epochs):
        perm = torch.randperm(len(yt))
        for i in range(0, len(yt), bs):
            idx = perm[i:i + bs]
            xb, yb = Xt[idx].to(DEVICE), yt[idx].to(DEVICE)
            loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return model.eval()

@torch.no_grad()
def predict_probs(model, X, bs=512):
    model.to(DEVICE).eval(); out = []
    Xt = torch.as_tensor(X)
    for i in range(0, len(Xt), bs):
        out.append(F.softmax(model(Xt[i:i + bs].to(DEVICE)), dim=1).cpu())
    return torch.cat(out).numpy().astype(np.float64)

WEIGHT_LAYERS = (nn.Conv2d, nn.Conv1d, nn.Linear)

def quantize_weights(model, bits):
    """Simulated symmetric per-output-channel weight quantization (weights only)."""
    m, qmax = copy.deepcopy(model), 2 ** (bits - 1) - 1
    for mod in m.modules():
        if isinstance(mod, WEIGHT_LAYERS):
            w = mod.weight.data; dims = tuple(range(1, w.dim()))
            scale = w.abs().amax(dim=dims, keepdim=True).clamp_min(1e-8) / qmax
            mod.weight.data = torch.round(w / scale).clamp(-qmax, qmax) * scale
    return m

def prune_global(model, sparsity):
    import torch.nn.utils.prune as prune
    m = copy.deepcopy(model)
    params = [(mod, "weight") for mod in m.modules() if isinstance(mod, WEIGHT_LAYERS)]
    prune.global_unstructured(params, pruning_method=prune.L1Unstructured, amount=sparsity)
    for mod, name in params: prune.remove(mod, name)
    return m

def finetune_masked(model, X, y, epochs, lr=5e-4, bs=256, seed=0):
    """Short fine-tune of a pruned model; zeroed weights stay zero (mask held fixed)."""
    set_seed(seed + 99); model.to(DEVICE).train()
    mods = [m for m in model.modules() if isinstance(m, WEIGHT_LAYERS)]
    masks = [(m.weight.data != 0).float() for m in mods]
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * math.ceil(len(y) / bs))
    Xt, yt = torch.as_tensor(X), torch.as_tensor(y).long()
    for ep in range(epochs):
        perm = torch.randperm(len(yt))
        for i in range(0, len(yt), bs):
            idx = perm[i:i + bs]
            loss = F.cross_entropy(model(Xt[idx].to(DEVICE)), yt[idx].to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            with torch.no_grad():
                for m, mk in zip(mods, masks): m.weight.data *= mk
    return model.eval()

def make_comps(Xtr, ytr, seed):
    ft = lambda s: (lambda m: finetune_masked(prune_global(m, s), Xtr, ytr, FT_EPOCHS, seed=seed))
    return {
        "fp32":       lambda m: m,
        "fp32+ft":    lambda m: finetune_masked(copy.deepcopy(m), Xtr, ytr, FT_EPOCHS, seed=seed),   # CONTROL
        "int8":       lambda m: quantize_weights(m, 8),
        "int4":       lambda m: quantize_weights(m, 4),
        "prune50":    lambda m: prune_global(m, 0.5),
        "prune50+ft": ft(0.5),
        "prune70+ft": ft(0.7),
    }

def mean_size(S, q):
    return float((S <= q).sum(1).mean())

def size_match_q(S_c, target_size, iters=40):
    """Label-free: smallest threshold whose mean set size on unlabeled inputs reaches target_size (bisection, size is monotone in q)."""
    lo, hi = 0.0, 1.0 + 1e-9
    for _ in range(iters):
        mid = (lo + hi) / 2
        if mean_size(S_c, mid) < target_size: lo = mid
        else: hi = mid
    return hi

## Data and experiment

In [ ]:
def load_medmnist(name, smoke):
    if smoke:
        rng = np.random.RandomState(0); K = 7
        def make(n):
            y = rng.randint(0, K, n); X = rng.rand(n, 3, 28, 28).astype(np.float32) * 0.8
            for k in range(K):
                r, c, ch = (k // 3) * 9 + 2, (k % 3) * 9 + 2, k % 3
                X[y == k, ch, r:r + 8, c:c + 8] += 0.6
            return (X - 0.5) / 0.5, y
        Xtr, ytr = make(2500); Xev, yev = make(1500)
        return Xtr, ytr, Xev, yev, K
    import medmnist
    from medmnist import INFO
    cls = getattr(medmnist, INFO[name]["python_class"])
    f = lambda a: ((a.imgs.astype(np.float32) / 255.0 - 0.5) / 0.5).transpose(0, 3, 1, 2).copy()
    lab = lambda a: a.labels.squeeze().astype(np.int64)
    tr, va, te = (cls(split=s, download=True) for s in ("train", "val", "test"))
    K = len(INFO[name]["label"])
    if name == "pathmnist":      # test split is from a different center: use test only (random halves)
        Xev, yev = f(te), lab(te)
    else:                        # random MedMNIST splits: pool val+test for more calibration data
        Xev, yev = np.concatenate([f(va), f(te)]), np.concatenate([lab(va), lab(te)])
    return f(tr), lab(tr), Xev, yev, K

def run_dataset(name):
    Xtr, ytr, Xev, yev, K = load_medmnist(name, SMOKE)
    print(f"[{name}] train {len(ytr)} | calib+eval pool {len(yev)} | classes {K} | class counts (pool) {np.bincount(yev, minlength=K).tolist()}")
    epochs = 3 if SMOKE else EPOCHS[name]
    rows = []
    for seed in SEEDS:
        t0 = time.time()
        model = train_model(ResNet18Small(K, width=16 if SMOKE else 64), Xtr, ytr, epochs, seed=seed)
        comps = make_comps(Xtr, ytr, seed)
        probs = {c: predict_probs(fn(model), Xev) for c, fn in comps.items()}
        print(f"  seed {seed} ({time.time()-t0:.0f}s) acc: " + " ".join(f"{c}={np.mean(probs[c].argmax(1)==yev):.3f}" for c in comps), flush=True)
        for sname, sfn in SCORES.items():
            S = {c: sfn(p) for c, p in probs.items()}
            for sp in range(N_SPLITS):
                rng = np.random.RandomState(1000 * seed + sp)
                perm = rng.permutation(len(yev)); cal, ev = perm[:len(yev) // 2], perm[len(yev) // 2:]
                q_full = conformal_q(S["fp32"][cal, yev[cal]])
                target = mean_size(S["fp32"][cal], q_full)          
                for c in comps:
                    q_c = conformal_q(S[c][cal, yev[cal]])
                    q_sm = size_match_q(S[c][cal], target)             
                    for mode, q in (("reuse", q_full), ("sizematch", q_sm), ("recal", q_c)):
                        r = summarize(S[c][ev] <= q, yev[ev], min_class_n=5)
                        rows.append(dict(dataset=name, seed=seed, split=sp, score=sname, comp=c, mode=mode,
                                         acc=float(np.mean(probs[c].argmax(1) == yev)), **r))
    df = pd.DataFrame(rows); df.to_csv(os.path.join(OUT_DIR, f"{name}_raw.csv"), index=False)
    return df

## Reporting

In [ ]:
from scipy import stats
COMP_ORDER = ["fp32", "fp32+ft", "int8", "int4", "prune50", "prune50+ft", "prune70+ft"]
def ci95(x):
    x = np.asarray(x, float); n = len(x)
    return float("nan") if n < 2 else stats.t.ppf(0.975, n - 1) * x.std(ddof=1) / np.sqrt(n)
def pm(x, f="{:+.2f}"): return (f + " +/- {:.2f}").format(np.mean(x), ci95(x))

def report(df):
    if SMOKE: print("*** SMOKE = True: SYNTHETIC DATA, NOT A RESULT ***")
    name = df["dataset"].iloc[0]
    per = df.groupby(["seed", "score", "comp", "mode"])[["coverage", "size", "worst_class_cov", "acc"]].mean().reset_index()
    P = {m: per[per["mode"] == m].set_index(["seed", "score", "comp"]) for m in ["reuse", "sizematch", "recal"]}
    t = pd.DataFrame({"acc": P["reuse"]["acc"],
        "gap_reuse": 100 * (P["reuse"]["coverage"] - (1 - ALPHA)), "gap_sm": 100 * (P["sizematch"]["coverage"] - (1 - ALPHA)),
        "gap_recal": 100 * (P["recal"]["coverage"] - (1 - ALPHA)),
        "sz_reuse": P["reuse"]["size"] / P["recal"]["size"], "sz_sm": P["sizematch"]["size"] / P["recal"]["size"],
        "wcc_reuse": P["reuse"]["worst_class_cov"], "wcc_sm": P["sizematch"]["worst_class_cov"], "wcc_recal": P["recal"]["worst_class_cov"]}).reset_index()
    rows = []
    for (sc, c), g in t.groupby(["score", "comp"]):
        rows.append(dict(score=sc, comp=c, acc=round(g["acc"].mean(), 3), gap_reuse=pm(g["gap_reuse"]), gap_sizematch=pm(g["gap_sm"]),
                         size_x_reuse=round(g["sz_reuse"].mean(), 2), size_x_sm=round(g["sz_sm"].mean(), 2),
                         wcc_reuse=round(g["wcc_reuse"].mean(), 3), wcc_sm=round(g["wcc_sm"].mean(), 3), wcc_recal=round(g["wcc_recal"].mean(), 3),
                         _o=COMP_ORDER.index(c), _gr=g["gap_reuse"].mean(), _gs=g["gap_sm"].mean()))
    o = pd.DataFrame(rows).sort_values(["score", "_o"])
    print(f"\n===== {name}: coverage gap in points vs 90 target (+/- 95% t-interval over {t['seed'].nunique()} seeds); size_x = set size / recal set size =====")
    print(o.drop(columns=["_o", "_gr", "_gs"]).to_string(index=False))
    return o

def summarize_all(outs):
    rows = [dict(dataset=n, score=r["score"], comp=r["comp"], reuse=r["_gr"], sizematch=r["_gs"]) for n, o in outs.items() for _, r in o.iterrows()]
    d = pd.DataFrame(rows)
    for m in ["reuse", "sizematch"]:
        p = d.pivot_table(index=["score", "comp"], columns="dataset", values=m).round(2)
        p = p.loc[[(s, c) for s in ["APS", "LAC"] for c in COMP_ORDER if (s, c) in p.index]]
        print(f"\n===== SUMMARY: mean coverage gap in points, method = {m} (negative = undercoverage) =====")
        print(p.to_string())
    print("\nQuestions this answers: (1) does fp32+ft under-cover like prune+ft? (control) (2) is |gap| for sizematch smaller than for reuse on every dataset?")

## Run

In [ ]:
outs, dfs = {}, []
for name in DATASETS:
    fpath = os.path.join(OUT_DIR, f"{name}_raw.csv")
    if os.path.exists(fpath) and not SMOKE:
        print(f"[{name}] found saved results, loading (delete {fpath} to rerun)"); df = pd.read_csv(fpath)
    else:
        df = run_dataset(name)
    dfs.append(df); outs[name] = report(df)
summarize_all(outs)
pd.concat(dfs).to_csv("step1_all_raw.csv", index=False)